# Question 3: Fitting a curve, and how well it predicts

Twelve measurements relate a setting $x$ to a response $y$. Five further measurements were
held back and never used for fitting.

You will fit polynomials of increasing degree, watch what happens to the error on the data
you fitted and on the data you held back, and then see what a penalty on the coefficients
does.

Save your own copy first (File > Save a copy in Drive). Then paste the INPUTS block from the
course page over the INPUTS cell below, replacing everything in that cell.

Run the cells in order and fill in every TODO. Use only `numpy` and `matplotlib`.

In [ ]:
# ===== INPUTS =====
# Replace this whole cell with the INPUTS block from the course page.
X_TRAIN = [-1.0, -0.82, -0.64, -0.45, -0.27, -0.09, 0.09, 0.27, 0.45, 0.64, 0.82, 1.0]
Y_TRAIN = None
X_HOLDOUT = [-0.85, -0.35, 0.15, 0.55, 0.92]
Y_HOLDOUT = None
DEGREE = 5
LAMBDA = 0.5
# ==================

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

if Y_TRAIN is None or Y_HOLDOUT is None:
    raise ValueError("Paste the INPUTS block from the course page over the INPUTS cell, "
                     "run that cell, then run this one again.")

x_tr = np.asarray(X_TRAIN, dtype=float)
y_tr = np.asarray(Y_TRAIN, dtype=float)
x_ho = np.asarray(X_HOLDOUT, dtype=float)
y_ho = np.asarray(Y_HOLDOUT, dtype=float)


def design(x, degree):
    """Given. Columns 1, x, x^2, ..., x^degree; the intercept column comes first."""
    return np.vander(np.asarray(x, dtype=float), degree + 1, increasing=True)


print(f"training points: {len(x_tr)}, held-out points: {len(x_ho)}")
print(f"design matrix at degree {DEGREE}: {design(x_tr, DEGREE).shape}")

### TODO 1: least squares by the normal equations

Complete `fit_ols` so that it returns the coefficient vector $\hat\beta$ minimizing
$\lVert y - X\beta \rVert^2$, by solving the normal equations

$$ (X^{\top} X)\, \hat\beta \; = \; X^{\top} y $$

Use `np.linalg.solve`. Do not use `np.polyfit` or `np.linalg.lstsq`.

In [ ]:
def fit_ols(X, y):
    """Least-squares coefficients from the normal equations."""
    # TODO 1: solve the normal equations above for beta
    return ...

### TODO 2: mean squared error

Complete `mse` so that it returns the mean squared error of the fit `beta` on the data
`(X, y)`:

$$ \mathrm{MSE} \; = \; \frac{1}{m} \sum_{i=1}^{m} \left( y_i - (X\beta)_i \right)^{2} $$

where $m$ is the number of rows of $X$.

In [ ]:
def mse(X, y, beta):
    """Mean squared error of the fit beta on (X, y)."""
    # TODO 2: return the mean of the squared residuals
    return ...

### TODO 3: least squares with a penalty

Ridge regression adds a penalty on the size of the coefficients, which pulls them toward
zero. The intercept is not penalized. The solution is

$$ \hat\beta_{\text{ridge}} \; = \; \left( X^{\top} X + \lambda P \right)^{-1} X^{\top} y $$

where $P$ is the identity with a zero in the intercept position. That matrix is built for
you below; complete the solve.

In [ ]:
def fit_ridge(X, y, lam):
    """Ridge coefficients, with the intercept left unpenalized."""
    penalty = np.eye(X.shape[1])
    penalty[0, 0] = 0.0           # don't penalize the intercept
    # TODO 3: solve the ridge equations above for beta, using `penalty`
    return ...


X_tr = design(x_tr, DEGREE)
X_ho = design(x_ho, DEGREE)

beta_line = fit_ols(design(x_tr, 1), y_tr)
beta_ols = fit_ols(X_tr, y_tr)
beta_ridge = fit_ridge(X_tr, y_tr, LAMBDA)

slope_line = beta_line[1]
mse_ols_holdout = mse(X_ho, y_ho, beta_ols)
mse_ridge_holdout = mse(X_ho, y_ho, beta_ridge)

print(f"degree 1  : slope = {slope_line:.4f}")
print()
print(f"{'fit':<22} {'train MSE':>10} {'holdout MSE':>12} {'|coefs| excl. intercept':>24}")
for name, beta in (("degree 1 OLS", beta_line),
                   (f"degree {DEGREE} OLS", beta_ols),
                   (f"degree {DEGREE} ridge", beta_ridge)):
    Xa = design(x_tr, len(beta) - 1)
    Xb = design(x_ho, len(beta) - 1)
    print(f"{name:<22} {mse(Xa, y_tr, beta):>10.4f} {mse(Xb, y_ho, beta):>12.4f}"
          f" {np.linalg.norm(beta[1:]):>24.4f}")

### Fitting higher capacity models

Each row fits a higher-degree polynomial to the same twelve training points and scores it
twice: on those same points, and on the five held-out points it never saw.

In [ ]:
print(f"{'degree':>7} {'parameters':>11} {'train MSE':>11} {'holdout MSE':>13}")
train_curve, holdout_curve = [], []
for d in range(1, 7):
    beta_d = fit_ols(design(x_tr, d), y_tr)
    tr = mse(design(x_tr, d), y_tr, beta_d)
    ho = mse(design(x_ho, d), y_ho, beta_d)
    train_curve.append(tr)
    holdout_curve.append(ho)
    print(f"{d:>7} {d + 1:>11} {tr:>11.4f} {ho:>13.4f}")

### Plotting the error of higher capacity models

The left panel plots the two error curves against degree. The right panel shows the three
fits over the data: the straight line, the degree $5$ least-squares curve, and the degree $5$
ridge curve. Hollow markers are the held-out points, which no fit ever saw.

In [ ]:
degrees = np.arange(1, 7)
grid = np.linspace(-1.1, 1.1, 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(degrees, train_curve, "o-", label="training MSE")
axes[0].plot(degrees, holdout_curve, "s--", label="held-out MSE")
axes[0].set(xlabel="polynomial degree", ylabel="mean squared error",
            title="Fitting error always falls. Prediction error does not.")
axes[0].legend()

axes[1].plot(x_tr, y_tr, "o", color="#1f77b4", label="training")
axes[1].plot(x_ho, y_ho, "o", markerfacecolor="none", markeredgecolor="black",
             markersize=9, label="held out")
axes[1].plot(grid, design(grid, 1) @ beta_line, label="degree 1")
axes[1].plot(grid, design(grid, DEGREE) @ beta_ols, label=f"degree {DEGREE} OLS")
axes[1].plot(grid, design(grid, DEGREE) @ beta_ridge, "--",
             label=f"degree {DEGREE} ridge")
axes[1].set(xlabel="x", ylabel="y", ylim=(-2.2, 3.4), title="The three fits")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

### Report your results

Run the cell below and enter R1, R2 and R3 on the course page.

In [ ]:
print("=" * 60)
print("QUESTION 3: REPORT VALUES")
print("=" * 60)
print(f"R1. Degree-1 slope (4 dec):                  {slope_line:.4f}")
print(f"R2. Degree-{DEGREE} OLS, held-out MSE (4 dec):     {mse_ols_holdout:.4f}")
print(f"R3. Degree-{DEGREE} ridge, held-out MSE (4 dec):   {mse_ridge_holdout:.4f}")

### Multiple-choice A

*This also appears on the course page, where you answer it. The options there may be in a different order.*

Look at the table in "Fitting higher capacity models". Which statement is best supported?

1. Both errors fall as the degree rises, so the highest degree tried is the best model available.
2. The degree with the lowest training MSE is the one that will predict new observations best.
3. Training MSE falls at every step, while held-out MSE is lowest at a low degree and stays above that lowest value at every higher degree.
4. Held-out MSE rises with the degree because a higher-degree curve passes further from the training points.

### Counterfactual

`LAMBDA_CF` sets how hard the penalty pulls the coefficients toward zero.

Change `LAMBDA_CF` from `0.5` to `50` in the cell below, re-run that cell only, and
compare its output with the degree-$5$ ridge row above. The multiple-choice question on the
course page asks what changed.

In [ ]:
LAMBDA_CF = 0.5     # <-- change to 50 and re-run this cell

beta_cf = fit_ridge(X_tr, y_tr, LAMBDA_CF)

print(f"LAMBDA_CF = {LAMBDA_CF}")
print(f"  training MSE            = {mse(X_tr, y_tr, beta_cf):.4f}")
print(f"  held-out MSE            = {mse(X_ho, y_ho, beta_cf):.4f}")
print(f"  |coefs| excl. intercept = {np.linalg.norm(beta_cf[1:]):.4f}")

### Multiple-choice B

*This also appears on the course page, where you answer it. The options there may be in a different order.*

You changed `LAMBDA_CF` from `0.5` to `50` and re-ran the cell. Which statement matches the
output?

1. The coefficients shrank further, training MSE rose, and held-out MSE rose too: the extra shrinkage went too far.
2. The coefficients shrank further, training MSE rose, and held-out MSE fell, because more shrinkage always predicts better.
3. The coefficients grew, because a larger penalty gives the fit more freedom.
4. Training MSE fell, because the penalty lets the curve follow the training points more closely.